# Avance 3: reporte del baseline

**Estimación de cambios de emisión de NOₓ por zona** en la Ciudad de México, combinando estaciones de monitoreo, el satélite TEMPO y un modelo físico de transporte.

Este cuaderno solo llama funciones de `nox_ml` (no tiene lógica propia) y se ejecuta de principio a fin sobre los datos reales (`dvc pull` antes). Las interpretaciones están escritas a partir de los resultados que se ven aquí.

In [ ]:
%matplotlib inline
from IPython.display import Markdown

from nox_ml import report
from nox_ml.data.inputs import load_inputs
from nox_ml.validation import compare, diagnostics, experiment

ROOT = report.find_root()
RUNS = ROOT / "runs"
config = report.load_config("baseline")
inputs = load_inputs(
    config.data_dir, config.period.start.isoformat(), config.period.end.isoformat(), config.simat_hours
)

## 1. El problema

**Emisión frente a concentración.** Lo que miden las estaciones y el satélite es *concentración* de NOₓ, que depende de cuánto se emite y de cómo está la atmósfera (viento y capa de mezcla). Un día con mucho NOₓ puede ser un día de más tráfico o un día sin viento. Lo que se quiere saber es la **emisión**: cuánto NOₓ sale de cada zona de la ciudad según el tipo de día (hábil, sábado, domingo o festivo), y cómo cambia.

**Objetivo.** Dado el NOₓ medido por estaciones y por satélite, más la meteorología, **estimar la emisión por zona y tipo de día** y sus cambios, con incertidumbre, separando el efecto de la emisión del efecto del clima. No existen mediciones directas de emisión por zona, así que no se puede entrenar contra ellas; por eso se validan los cambios con un caso cuya respuesta se conoce: el domingo, cuando el tráfico contado en las calles baja a 0.65-0.73 de un día hábil.

**Datos.** Las tardes de noviembre y diciembre 2023, de 12 a 17 h locales, que es cuando TEMPO escanea la ciudad:

In [ ]:
report.data_overview(inputs, config)

In [ ]:
report.plot_domain(inputs)

**Lectura del mapa.** Las 24 estaciones cubren el centro y el poniente del dominio con densidad; el noreste y el este (zonas 5 y 8 de la rejilla 3×3) tienen pocas. TEMPO, en cambio, da píxeles en todo el dominio, incluidas las zonas sin estación. Las líneas punteadas son las 9 zonas en las que la inversión estima un factor de emisión de las calles.

In [ ]:
report.plot_daily_context(inputs)

**Lectura de la serie.** El NOₓ de la tarde varía de unos 10 a más de 70 ppb entre días, y buena parte de esa variación acompaña al viento y a la capa de mezcla (los días de poco viento y capa baja de mediados de diciembre son los más altos). Los domingos y festivos (naranja) suelen estar por debajo, pero no siempre: un domingo con poco viento puede tener más NOₓ que un día hábil ventilado. Mirar la concentración sola no separa la emisión del clima; ese es el trabajo del modelo.

## 2. Algoritmo: inversión bayesiana lineal con pluma estacionaria

**Cómo funciona.** Una pluma gaussiana estacionaria, con el viento y la capa de mezcla de cada hora, dice cuánto NOₓ llega a cada estación y a cada píxel de TEMPO por cada tonelada emitida en cada fuente. Como la concentración es **lineal en la emisión**, el problema es una regresión lineal bayesiana:

- **Incógnitas (31):** 4 fuentes puntuales (2 centrales eléctricas y 2 aeropuertos) y un factor multiplicativo por zona (3×3 = 9) y tipo de día (hábil, sábado, domingo/festivo) sobre la emisión a priori de las calles, repartida con el peso de la clase de calle en OpenStreetMap.
- **A priori:** cada factor vale 1 ± 0.5, igual para los tres tipos de día, de modo que cualquier diferencia entre ellos viene de los datos.
- **Observaciones:** NOₓ de las estaciones (error de 15 ppb) y columnas de NO₂ de TEMPO; el fondo de cada hora se integra analíticamente.
- **Resultado:** una posterior gaussiana en forma cerrada (media y desviación de cada emisión) y un intervalo de los cocientes entre tipos de día por bootstrap sobre días completos.

**Por qué es apropiado:**

| Necesidad del problema | Qué aporta la inversión bayesiana con física |
|---|---|
| No hay etiquetas de emisión | Estima la emisión como incógnita de un modelo de transporte; no necesita ejemplos de emisión conocida |
| Pocos datos (~24 estaciones, 61 días) | La física fija la forma de la relación y el a priori regulariza; solo hay 31 parámetros |
| Se necesita incertidumbre | La posterior da desviación por incógnita y el bootstrap da intervalos de los cocientes |
| Debe poder interpretarse | Cada parámetro es un factor de emisión de una zona y tipo de día, en toneladas por año |
| Separar emisión de clima | El viento y la capa de mezcla entran en el modelo directo, así que el clima se explica con la física y lo que queda se asigna a la emisión |

**IDW como referencia de concentración.** La interpolación por distancia inversa predice el NOₓ de una estación oculta con el promedio de las demás estaciones de la misma hora, ponderado por 1/distancia². Es un buen termómetro de cuánta concentración se puede predecir solo con las vecinas, pero **no puede responder la pregunta**: no tiene noción de emisión, de viento ni de tipo de día, y por tanto no puede decir cuánto baja la emisión un domingo. Aparece en las tablas como piso de concentración, no como candidato.

## 3. Corrida del baseline

Se ejecuta `configs/experiments/baseline.yaml` con el corredor del proyecto (equivale a `nox-ml run`): ajusta el modelo, lo valida dejando una estación fuera a la vez, estima los cocientes de emisión por tipo de día y calcula el placebo (domingos falsos entre días hábiles).

In [ ]:
out = experiment.run_experiment(config, RUNS)
metrics = report.load_metrics(out)
print(experiment.headline(metrics))

In [ ]:
table, _ = compare.collect([out])
Markdown(compare.markdown(table))

**Lectura.** La corrida reproduce la fila de referencia de la investigación: r de 0.77 con las estaciones y 0.74 con TEMPO (dentro de muestra), error en estación oculta de 13.6 ppb para el modelo, 11.6 para IDW y 11.7 para el híbrido, cociente del domingo de 0.42 y placebo centrado cerca de 1. El ajuste completo tarda unos 12-14 s.

- `loo_rmse_ppb`: error al predecir una estación que el modelo no vio.
- `sunday_ratio`: emisión del domingo sobre la del día hábil, con su intervalo del 90 %.
- `placebo_center` y `placebo_halfwidth`: qué cociente sale cuando no hay cambio, y qué tan ancho es. `eligible` indica si pasa la compuerta del placebo.

In [ ]:
report.plot_loo_by_station(metrics)

**Lectura.** El modelo de inversión pierde frente a IDW en la mayoría de las estaciones (gana solo en 7 de 24) y pierde por más de 4 ppb en las del centro y sur (PED, CCA, HGM, IZT, CAM, MGH), donde las vecinas cercanas ya dicen casi todo. Gana en estaciones de la periferia (MON, ACO, VIF, CHO, CUA, ATI, TLI), donde IDW no tiene vecinas cercanas. El híbrido, que suma al modelo el IDW de sus residuos, queda cerca de IDW y le gana en 15 de 24 estaciones; eso indica que la física aporta algo que las vecinas no tienen, pero que el modelo solo no captura el nivel de cada estación.

## 4. Métrica

**Principal: `loo_rmse_ppb`**, el error cuadrático medio (en ppb) al predecir una estación que se dejó fuera del ajuste. Mide justo lo que importa: si la física y las emisiones estimadas explican lo que se mide *donde no hay sensor*. Está en las unidades que la gente lee y es la misma para modelos que estiman emisión y para las referencias de concentración.

**Secundarias:**

- **Domingo frente a día hábil (`sunday_ratio`)** y su diferencia con el tráfico contado (0.65-0.73): es un cambio de emisión con respuesta conocida, el único control directo de emisión que hay.
- **Placebo (`placebo_center`, `placebo_halfwidth`):** el mismo cociente cuando se etiquetan como "domingo" días hábiles al azar, donde no hay cambio que encontrar. Debe centrarse en 1 ± 0.15; su semiancho es el cambio mínimo detectable.
- **Sesgo frente a ritmo (`loo_bias_ppb`, `loo_hourly_rmse_ppb`):** separa el error en el nivel de cada estación (lo que se repite todas las horas) del error de hora en hora.
- **`r_tempo`:** correlación del modelo con las columnas de NO₂ de TEMPO.

In [ ]:
report.plot_level_and_rhythm(metrics)

**Lectura.** Los tres predictores tienen un error de nivel por estación de 6-7 ppb y uno de hora en hora de 9-12 ppb. El modelo pierde frente a IDW sobre todo en el ritmo (11.8 contra 9.5 ppb): IDW sigue el clima de la hora a través de las otras estaciones, mientras que el modelo usa un viento promedio de todo el dominio.

In [ ]:
report.plot_emission_metrics(metrics)

**Lectura.** El domingo cae a 0.42 (intervalo 0.35-0.54), por debajo del 0.65-0.73 que dicen los conteos de tráfico: el intervalo completo está fuera del rango esperado. El placebo queda centrado en 1.08, dentro de la compuerta de 1 ± 0.15, pero con un semiancho de 0.35: con este diseño solo se detectan cambios de más o menos un tercio.

**Por qué no R².** Un R² global mezcla tres fuentes de variación que el problema no quiere mezclar: las diferencias de nivel entre estaciones, el ciclo del clima de hora en hora y la señal de emisión. Un modelo puede tener un R² alto solo por seguir el clima y aun así asignar mal la emisión; aquí pasa eso: con r = 0.77 contra las estaciones, el cociente del domingo queda 0.23 fuera del rango de tráfico. Además es adimensional: no dice si el error es de 2 o de 20 ppb. Por eso el error en ppb de la estación oculta es la métrica principal, y la respuesta del domingo y el placebo miden la parte de emisión.

**`r_tempo` es dentro de muestra.** TEMPO se usa en el ajuste, así que su correlación no es una validación independiente. Pero con tan pocos parámetros no puede ser memorización:

In [ ]:
report.capacity_table(metrics)

El modelo tiene 31 parámetros para unos 78,000 píxeles (más de 2,500 píxeles por parámetro): no tiene capacidad para ajustar píxel por píxel, así que `r_tempo` mide si el **patrón espacial** de NO₂ que implican las emisiones estimadas coincide con el que ve el satélite. Lo honesto, sin embargo, es el error en estación oculta, que sí se calcula con datos que el ajuste no vio.

## 5. Características importantes

Las "características" del modelo son las **observaciones**. Se analizan de dos maneras, con `nox-ml diagnose` (`nox_ml.validation.diagnostics`):

- **Información por tipo de observación (embedded).** Como la posterior es gaussiana, lo que las observaciones dicen de cada incógnita no depende de sus valores, solo de la matriz Jᵀ R⁻¹ J. Se calcula la fracción de la varianza a priori que elimina cada fuente (solo estaciones, solo TEMPO, ambas); la suma sobre las incógnitas son los grados de libertad para la señal (DOFS).
- **Ablación (wrapper).** Se reajusta el baseline sin TEMPO, con 15 y 9 estaciones (5 subconjuntos al azar pero repartidos en el dominio) y con el viento girado 15° y 30°.

Las tres etapas se corren por separado (cada una escribe sus tablas en `runs/baseline/diagnostics/`; en total, unos 6-8 minutos).

In [ ]:
diagnostics.run_diagnostics(config, RUNS, stages=[diagnostics.Stage.information], echo=print)

In [ ]:
diagnostics.run_diagnostics(config, RUNS, stages=[diagnostics.Stage.ablation], echo=print)

In [ ]:
diagnostics.run_diagnostics(config, RUNS, stages=[diagnostics.Stage.learning], echo=print)

In [ ]:
tables = report.load_diagnostics(out)
report.plot_information(tables["reduction"], tables["dofs"])

**Lectura de la información.** TEMPO por sí solo aporta 31.0 de los 31 grados de libertad; las estaciones solas, 26.2; las dos juntas, 31.0: las estaciones no añaden información a la que ya da TEMPO. Donde más faltan las estaciones es en las zonas 5 y 8 (este y noreste, las que el mapa mostraba casi vacías), sobre todo en sábado y domingo (reducción de 0.2-0.5) y en el aeropuerto AIFA (0.56). La reducción de varianza de TEMPO (≈ 1) es un techo optimista, porque supone errores independientes entre píxeles; por eso el intervalo del cociente del domingo se calcula con bootstrap de días, y es más ancho que lo que sugiere esa varianza.

In [ ]:
report.plot_ablation(tables["ablation"])

**Lectura de la ablación.**

- **Sin TEMPO** el error en estación oculta sube de 13.6 a 14.6 ppb, el placebo se ensancha (0.35 → 0.43) y el cociente del domingo **sube de 0.42 a 0.80**, ya por encima del rango de tráfico. La respuesta del domingo depende mucho de TEMPO: con él es demasiado baja y sin él demasiado alta; lo correcto queda en medio, lo que apunta a que el peso del satélite necesita ajuste.
- **Menos estaciones** empeora el error en estación oculta (13.6 con 24; 13.9 con 15; 15.3 con 9, con más dispersión entre subconjuntos), pero **no mueve** el cociente del domingo, el placebo ni `r_tempo`. Las emisiones estimadas las determina TEMPO; las estaciones sirven para validar y para el nivel de fondo.
- **Viento girado** 15° y 30°: no empeora nada (13.5 y 13.3 ppb; domingo 0.41 y 0.40). El viento promedio del dominio es una entrada tosca y, con él, este baseline no es sensible a su dirección: no se puede afirmar que mejorarlo ayudaría solo por esta prueba.

## 6. Sub y sobreajuste

**Dentro de muestra frente a estación oculta.** Si el modelo estuviera sobreajustado, el error con la estación oculta sería mucho mayor que el error dentro de muestra. Se compara por estación el RMSE del modelo ajustado con todas las estaciones con el del ajuste que deja esa estación fuera:

In [ ]:
held = report.in_sample_vs_held_out(inputs, config, metrics)
report.plot_in_sample_vs_held_out(held)

In [ ]:
held.describe().loc[["mean", "min", "max"]].round(2)

**Lectura.** Los puntos están casi sobre la diagonal: dentro de muestra el RMSE es de 13.0 ppb contra 13.6 ppb con la estación oculta en conjunto (12.6 contra 13.2 en el promedio por estación, tabla de arriba), una diferencia de unos 0.6 ppb. **No hay sobreajuste** (con 31 parámetros no podía haberlo). El problema es el contrario: el error es alto aun dentro de muestra, así que hay **subajuste**; el modelo no alcanza a explicar la concentración ni de las estaciones que sí ve (el viento único del dominio, la pluma estacionaria y las zonas 3×3 son demasiado simples). Las estaciones MER, TLA, SAG y XAL son las más difíciles, con errores de 16-19 ppb aun dentro de muestra.

**Curvas de aprendizaje.** Se repite el ajuste con menos días de datos (bloques de días consecutivos) y con menos estaciones (subconjuntos repartidos); la línea es la media de 5 repeticiones y la banda su rango:

In [ ]:
report.plot_learning_curves(tables["curve"])

**Lectura.**

- **Por días:** el error en estación oculta es prácticamente el mismo con 30, 45 o 61 días (13.3, 13.2 y 13.6 ppb, dentro de la dispersión entre bloques). Más días no ayudarían a este modelo; con 15 días el error sube y se dispersa más (14.1 ppb, rango 11-16).
- **Por estaciones:** el error en estación oculta baja de 15.3 ppb con 6 estaciones a 13.6 con 24, mientras que el error dentro de muestra casi no cambia (13.0). La brecha entre ambos se cierra de 2.3 a 0.6 ppb y la curva todavía baja al llegar a 24: el modelo se beneficia de más estaciones, y IDW todavía más (de 14.7 a 11.6 ppb).
- El comportamiento es el de un modelo con **sesgo alto y varianza baja**: un piso de error que no se corrige con más días. Mejorarlo exige más capacidad (viento por zona, pluma que dependa de la hora, mejor a priori), no más datos de este mismo tipo.

## 7. Desempeño mínimo y conclusiones

El desempeño mínimo del baseline, con los criterios del plan:

- **Placebo** centrado en 1 ± 0.15 (compuerta de elegibilidad).
- **Dirección del domingo**: la emisión del domingo debe quedar por debajo de la del día hábil.
- **Cercanía a IDW** en concentración: no quedar muy por debajo. Aquí se toma "muy por debajo" como más de 25 % peor en RMSE que IDW (umbral de este reporte, no una regla del proyecto).

Se agrega la magnitud del domingo frente al tráfico contado como referencia, sin que sea parte del mínimo.

In [ ]:
report.minimum_performance(metrics)

**Conclusión.**

El baseline **pasa el desempeño mínimo**: el placebo queda centrado en 1.08 (dentro de 1 ± 0.15, con semiancho 0.35), ve la dirección correcta del domingo (0.42, intervalo 0.35-0.54, todo por debajo de 1) y su error en estación oculta (13.6 ppb) está 17 % por encima de IDW (11.6 ppb).

Pero **no es un buen modelo todavía**:

1. **Exagera la baja del domingo:** 0.42 frente a 0.65-0.73 del tráfico contado; el intervalo completo queda fuera del rango, a 0.23 de él. Con el placebo en 0.35 de semiancho, el modelo no podría distinguir cambios pequeños aunque acertara el centro.
2. **Depende de TEMPO de forma crítica:** sin él, el cociente sube a 0.80 (por encima del rango); con él es demasiado bajo. La verdad queda entre los dos, y el peso del satélite es una perilla sin calibrar.
3. **No supera a IDW:** 13.6 contra 11.6 ppb; solo le gana en 7 de 24 estaciones. IDW no estima emisión, pero marca el piso de concentración que un modelo de emisión debe igualar, y este baseline no lo alcanza por sí solo (el híbrido modelo + IDW de residuos sí queda a la par, 11.7).
4. **Subajusta:** el error es alto aun dentro de muestra, y más días de datos no lo reducen.

**Qué deben mejorar los modelos del avance 4:**

- Bajar el error en estación oculta hacia el nivel de IDW o por debajo (≤ 11.6 ppb), sobre todo en el ritmo de hora en hora, con un viento menos tosco que el promedio del dominio o con transporte con memoria (puffs, vida química).
- Acercar el cociente del domingo al rango de tráfico (0.65-0.73) sin salirse de él cuando se quita TEMPO: ajustar el peso del satélite y agregar una restricción de tráfico.
- Mantener el placebo centrado en 1 ± 0.15 (elegibilidad) y bajar su semiancho para detectar cambios menores.
- Dar más información en las zonas con pocas estaciones (este y noreste) donde hoy solo TEMPO determina la emisión.